# Recipe 02 — Exporters
> Ship traces and spans to consoles, files, or OpenTelemetry collectors.

| | |
|---|---|
| **Module** | `anchor.observability.exporters` |
| **Key classes** | `ConsoleSpanExporter`, `InMemorySpanExporter`, `FileSpanExporter`, `OTLPSpanExporter` |
| **Difficulty** | Beginner |

In [ ]:
# Setup
from anchor.observability import Tracer, SpanKind
from anchor.observability.exporters import (
    ConsoleSpanExporter,
    InMemorySpanExporter,
    FileSpanExporter,
    OTLPSpanExporter,
)
import time

## 1 — Generate sample trace data

In [ ]:
tracer = Tracer()
trace = tracer.start_trace("export-demo", attributes={"pipeline": "rag"})

span1 = tracer.start_span(trace.trace_id, "retrieval", kind=SpanKind.RETRIEVAL, attributes={"top_k": 10})
time.sleep(0.01)
tracer.end_span(span1)

span2 = tracer.start_span(trace.trace_id, "reranking", kind=SpanKind.RERANKING, parent_span_id=span1.span_id)
time.sleep(0.01)
tracer.end_span(span2)

trace = tracer.end_trace(trace)
print(f"Trace ready: {len(trace.spans)} spans")

## 2 — Exporter walkthrough

In [ ]:
# ConsoleSpanExporter — prints spans to stdout
console = ConsoleSpanExporter()
console.export(trace.spans)
print("\n(Spans printed above by ConsoleSpanExporter)")

In [ ]:
# InMemorySpanExporter — stores for programmatic inspection
in_memory = InMemorySpanExporter()
in_memory.export(trace.spans)

stored = in_memory.get_spans()
print(f"Stored spans: {len(stored)}")
for s in stored:
    print(f"  {s.name}: kind={s.kind}")

In [ ]:
# FileSpanExporter — writes JSON to disk
import tempfile, os, json

tmp_path = os.path.join(tempfile.gettempdir(), "anchor_traces.json")

file_exp = FileSpanExporter(file_path=tmp_path)
file_exp.export(trace.spans)

size = os.path.getsize(tmp_path)
print(f"Wrote {size} bytes to {tmp_path}")

In [ ]:
# Peek at the file contents
with open(tmp_path) as f:
    data = json.load(f)
print(f"Span records in file: {len(data)}")
for record in data:
    print(f"  {record.get('name', 'unknown')}")

In [ ]:
# OTLPSpanExporter — sends spans over gRPC/HTTP to an OTLP endpoint
# Uncomment when you have a collector running:
#
# otlp = OTLPSpanExporter(endpoint="http://localhost:4317")
# otlp.export(trace.spans)

print("OTLPSpanExporter pattern:")
print('  otlp = OTLPSpanExporter(endpoint="http://localhost:4317")')
print('  otlp.export(trace.spans)')

## 3 — Combining multiple exporters

In [ ]:
exporters = [
    ConsoleSpanExporter(),
    InMemorySpanExporter(),
]

for exporter in exporters:
    exporter.export(trace.spans)
    print(f"Exported to {type(exporter).__name__}")

In [ ]:
# Verify in-memory exporter accumulated spans
mem_exporter = exporters[1]
print(f"InMemory exporter holds {len(mem_exporter.get_spans())} spans")

## Key Takeaways
- `ConsoleSpanExporter` is the fastest way to see spans during development.
- `InMemorySpanExporter` is perfect for tests and assertions.
- `FileSpanExporter` writes JSON for offline analysis.
- `OTLPSpanExporter` integrates with Jaeger, Grafana Tempo, and other OTLP backends.

**Next:** [Cost Tracking](03_cost_tracking.ipynb)